# ストレージ

本章では、AWSソリューションアーキテクトアソシエイト試験に必要なストレージサービスについて学習します。

## 本章の構成

- ブロックストレージ
- ファイルストレージ
- オブジェクトストレージ
- ハイブリッドストレージ
- 高性能ファイルシステム

### 本章のサービス一覧

| | サービス | ひとこと |
|---|----------|----------|
| <img src="images/aws-icons/ebs.png" width="32" alt="Amazon EBS"/> | **Amazon EBS** | EC2用ブロックディスク |
| <img src="images/aws-icons/efs.png" width="32" alt="Amazon EFS"/> | **Amazon EFS** | Linux向け共有ファイル（NFS） |
| <img src="images/aws-icons/s3.png" width="32" alt="Amazon S3"/> | **Amazon S3** | オブジェクトストレージ |
| <img src="images/aws-icons/storage-gateway.png" width="32" alt="AWS Storage Gateway"/> | **AWS Storage Gateway** | オンプレとクラウドストレージの橋 |
| <img src="images/aws-icons/data-sync.png" width="32" alt="AWS DataSync"/> | **AWS DataSync** | オンプレ↔AWSのオンライン同期 |
| <img src="images/aws-icons/snowball.png" width="32" alt="AWS Snowball"/> | **AWS Snowball** | 大容量データのオフライン搬送 |
| <img src="images/aws-icons/aws-backup.png" width="32" alt="AWS Backup"/> | **AWS Backup** | 複数サービスのバックアップ一元管理 |
| <img src="images/aws-icons/fsx.png" width="32" alt="Amazon FSx"/> | **Amazon FSx** | Windows/Lustre等の高性能ファイル |

### ストレージの選び方（早見）

| 種類 | 代表サービス | イメージ |
|------|--------------|----------|
| ブロック | EBS | 1台のVM用ディスク |
| ファイル | EFS / FSx | 複数EC2で共有フォルダ |
| オブジェクト | S3 | 画像・ログ・静的Web |

## ブロックストレージ

**この章で覚えること**

- **EBS** が EC2 用ディスク（**インスタンスストア**は一時用）
- スナップショット
- 暗号化
- Multi-Attach

**Amazon Elastic Block Store (Amazon EBS)** は、EC2インスタンスにアタッチして使用する高性能で低レイテンシのブロックストレージサービスです。新規作成では**gp3**を優先し、レガシーとして**gp2**も残っています。
その他、プロビジョンドIOPS SSD（**io1/io2**）、スループット最適化HDD（**st1**）、Cold HDD（**sc1**）があります。

**インスタンスストア**はEC2に付属する一時ディスクで、停止・障害で消える場合があります。高速キャッシュ向けで、永続データには**EBS**を使います。

EBSの主なユースケースには、データベースのストレージや高パフォーマンスが求められるアプリケーションのバックエンドが含まれます。例えば、**io1/io2**は、トランザクションが頻繁に発生するデータベースに適しています。
一方、**st1**や**sc1**は、ログデータやバックアップデータなど、大容量のデータ保存に適しています。

EBSの重要な機能として、**スナップショット**があります。これは、データのバックアップを取得し、別のリージョンに複製することが可能で、災害復旧に役立ちます。
また、EBSは**データの暗号化**をサポートしており、データのセキュリティを強化します。**Multi-Attach**（主にio2）は、1つのEBSを複数EC2から同時マウントするクラスタ向け機能です。

---

## ファイルストレージ

**この章で覚えること**

- **EFS** で複数 EC2 から NFS 共有

**Amazon Elastic File System (Amazon EFS)** は、完全マネージド型のNFSファイルシステムを提供するサービスです。EFSは、複数のEC2インスタンスから同時にアクセスでき、スケーラブルで高い可用性を持つストレージです。

EFSの主なユースケースには、コンテンツ管理システムや、ホームディレクトリの共有ストレージが含まれます。例えば、ウェブサーバーが複数台ある場合、これらが同じデータにアクセスする必要がある時にEFSは非常に有効です。
EFSは**自動的に容量を拡張または縮小**するため、使用量に応じてストレージを最適化できます。

EFSには、**標準ストレージクラス**と**インフリークエントアクセスストレージクラス**があり、頻繁にアクセスされるデータとまれにアクセスされるデータを分けることで、コストを最適化できます。

---

## オブジェクトストレージ

**この章で覚えること**

- **S3**
- ストレージクラス・ライフサイクル
- CRR
- Transfer Acceleration
- Object Lock 等

**Amazon S3** は、スケーラブルで高可用性を持つオブジェクトストレージサービスです。バケット単位でオブジェクト（ファイル）を管理します。
主な機能は**バージョニング**、**ライフサイクルポリシー**、**バケットポリシー**（アクセス制御の主流）です。ACLもありますが、新規設計ではバケットポリシーとIAMを優先するのが一般的です。

| ストレージクラス | 用途の目安 |
|------------------|------------|
| S3 Standard | 頻繁アクセス、データレイク |
| S3 Standard-IA | 月に数回程度のアクセス |
| S3 Glacier Instant Retrieval | 四半期に1回程度、即時取得したいアーカイブ |
| S3 Glacier Flexible / Deep Archive | 長期保管。取得に時間がかかる |

**S3 Intelligent-Tiering**は、アクセス頻度に応じてクラスを自動で切り替え、コスト最適化したいときに使います。

**S3の試験で押さえる機能**

- **耐久性**：設計上99.999999999%（11個の9）の耐久性
- **クロスリージョンレプリケーション（CRR）**：別リージョンへ非同期複製（DR・低RPO）
- **Transfer Acceleration**：遠方からのアップロード高速化
- **Object Lock**：コンプライアンス向けの削除防止（WORM）

- **向く例**：静的Web、バックアップ、ログ蓄積、分析用データレイク
- **向かない例**：複数EC2が同時にPOSIXファイルとして共有 → EFS を検討

**S3 Glacier** は、上記のとおり **S3のストレージクラス**（独立サービスというより低頻度・アーカイブ向けクラス）です。

---

## ハイブリッドストレージ

**この章で覚えること**

- **Storage Gateway**
- **DataSync**
- **Snowball**
- **AWS Backup**

**AWS Storage Gateway**は、オンプレと AWS ストレージをつなぐハイブリッドサービスです。

- **ファイルゲートウェイ**：NFS/SMB 共有
- **ボリュームゲートウェイ**：iSCSI ボリューム
- **テープゲートウェイ**：バックアップ・アーカイブ

> **関連サービス（移行・バックアップ）**
>
>

> | サービス | 用途 |
> |----------|------|
> | **AWS DataSync** | オンラインでオンプレ↔AWS間を高速同期 |
> | **AWS Snowball / Snowmobile** | 大容量データのオフライン搬送 |
> | **AWS Backup** | EBS・RDS・EFS等のバックアップをポリシーで一元管理 |
---

## 高性能ファイルシステム

**この章で覚えること**

- **FSx**（Windows / Lustre / ONTAP 等）

**Amazon FSx**は、用途別の高性能マネージドファイルシステムです。

| 種類 | 向く例 |
|------|--------|
| **Windows File Server** | Windows アプリ、AD 連携 |
| **Lustre** | HPC、ビッグデータ |
| **NetApp ONTAP / OpenZFS** | 既存 ONTAP / ZFS ワークロード |

例えば、FSx for Windows File Serverは、Windowsベースのアプリケーションに対して高い互換性を提供し、Active Directoryと統合可能です。一方、FSx for Lustreは、高性能コンピューティング（HPC）やビッグデータ解析のワークロードに適しています。
FSx for NetApp ONTAPは、NetAppの機能をクラウドで利用するために最適です。

> ### 試験で押さえる設計の考え方（本章関連）
>
>

> - **耐久性・DR**：S3 CRR、EBSスナップショット、AWS Backup
> - **コスト**：S3ライフサイクル・ストレージクラス・Intelligent-Tiering
## まとめ

- **EBS**：EC2用ブロックストレージ。スナップショット・暗号化。インスタンスストアは一時用
- **S3**：CRR・Transfer Acceleration・Object Lock（試験補足）
- **DataSync / Snowball / AWS Backup**：移行・バックアップ統合
- **EFS**：Linux向け共有NFS。Webサーバー複数台の共有に便利
- **S3**：オブジェクトストレージ。クラスとライフサイクルでコスト最適化
- **Storage Gateway**：オンプレとS3/EBS等をつなぐハイブリッド
- **FSx**：Windows / Lustre / NetApp など用途別の高性能ファイル


## 復習クイズ

> **取り組み方**
> 1. 下の「問題」だけを読み、口頭またはメモで答える
> 2. すべて答えたあと、**区切りセル**まで進み、確認してから**解答・解説セル**へ進む
> 3. 解答が見えている場合は、紙などで隠してから取り組む

### 問題

**Q1.** LinuxのWebサーバーを複数台立て、同じディレクトリを同時にマウントして共有したい。最も適切なストレージサービスは何か。

**Q2.** EC2に付属し、インスタンス停止や障害でデータが失われることがあるディスクの種類は何か。

**Q3.** 新規に作成する汎用SSDのEBSボリュームで、現在推奨されるボリュームタイプは何か。

**Q4.** オブジェクトを月に1回程度しか読まない。コストを抑えつつS3に置き続けたい。検討すべきストレージクラスの例は何か。

**Q5.** バケット単位でファイル（オブジェクト）を管理する、AWSのオブジェクトストレージサービス名は何か。

**Q6.** オンプレミスのファイルサーバーから、AWSへ段階的にバックアップしたい。ハイブリッド接続に使うサービスは何か。

**Q7.** 東京リージョンのS3バケットを、災害対策で大阪リージョンへ非同期複製したい。使う機能名は何か。


---

## ここまでで回答を考えてください

すべての問題（Q1〜Q7）に、口頭またはメモで答えてから、**次のセル**へ進んでください。

解答・解説は次のセルにあります。まだ答えを考えていない場合は、下にスクロールしないでください。

---


## 復習クイズ　解答・解説

**A1.** **Amazon EFS**  
複数EC2から同時マウントできるNFS。S3はオブジェクトストアでPOSIX共有には不向き。→ ファイルストレージ

**A2.** **インスタンスストア**  
一時・高速だが永続しない。永続データはEBS。→ ブロックストレージ

**A3.** **gp3**  
新規汎用SSDはgp3優先。gp2はレガシー寄り。→ ブロックストレージ

**A4.** **S3 Standard-IA**（または Glacier Instant Retrieval など低頻度向けクラス）  
アクセス頻度に応じたクラス選択。→ オブジェクトストレージ

**A5.** **Amazon S3**  
オブジェクトストレージの代表。→ オブジェクトストレージ

**A6.** **AWS Storage Gateway**  
オンプレとS3/EBS等をつなぐ。ファイル/ボリューム/テープの3モード。→ ハイブリッドストレージ

**A7.** **クロスリージョンレプリケーション（CRR）**  
別リージョンへ非同期複製（DR）。→ オブジェクトストレージ（S3の試験で押さえる機能）
